### Preámbulo

In [1]:
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

import $ivy.$                                

import org.scalatest.{Filter => _, _}, flatspec._, matchers._


# Tema 7. Aplicaciones

## 7.2 Implementar y consultar modelos de datos

La biblioteca de colecciones de Scala facilita enormemente la implementación y consulta de modelos de datos. Por ejemplo, las siguientes clases modelan la estructura de una organización formada por departamentos, empleados y tareas que los empleados pueden realizar.

In [3]:
// departamentos

case class Department(id: Department.Id)
object Department:
    type Id = String

// Empleados

case class Employee(id: Employee.Id, dpt: Department.Id)
object Employee:
    type Id = String

// tareas

case class Task(id: Task.Id, hours: Int)
object Task:
    type Id = String

// La organización completa

case class Organization(
    departments: Map[Department.Id, Department], 
    employees: Map[Employee.Id, Employee], 
    tasks: Map[Task.Id, Task],
    knows: List[(Employee.Id, Task.Id)])

defined class Department
defined object Department
defined class Employee
defined object Employee
defined class Task
defined object Task
defined class Organization

Esta implementación es un ejemplo de modelo de datos _plano_. La característica clave de este tipo de modelos es que las distintas entidades (empleados, departamentos y tareas, en este caso) se refieren unas a otras usando _claves_. Esta es una posible instancia del modelo de datos de la organización:

In [4]:
val org: Organization = Organization(
    Map(
        "Product"  -> Department("Product"),
        "Quality"  -> Department("Quality"),
        "Research" -> Department("Research"),
        "Sales"    -> Department("Sales")),
    
    Map("Alex"     -> Employee("Alex", "Product"), 
        "Bert"     -> Employee("Bert", "Product"), 
        "Cora"     -> Employee("Cora", "Research"), 
        "Drew"     -> Employee("Drew", "Research"), 
        "Edna"     -> Employee("Edna", "Research"), 
        "Fred"     -> Employee("Fred", "Sales")),
    
    Map("build"    -> Task("build", 3), 
        "abstract" -> Task("abstract", 5), 
        "design"   -> Task("design", 2),
        "call"     -> Task("call", 1),
        "program"  -> Task("program", 3)),
    
    List(
        ("Alex", "build"),
        ("Bert", "build"),
        ("Cora", "abstract"),
        ("Cora", "build"),
        ("Cora", "design"),
        ("Drew", "abstract"),
        ("Drew", "design"),
        ("Edna", "abstract"),
        ("Edna", "call"),
        ("Edna", "design"),
        ("Fred", "call")))

org: Organization = Organization(
  departments = Map(
    "Product" -> Department(id = "Product"),
    "Quality" -> Department(id = "Quality"),
    "Research" -> Department(id = "Research"),
    "Sales" -> Department(id = "Sales")
  ),
  employees = HashMap(
    "Alex" -> Employee(id = "Alex", dpt = "Product"),
    "Fred" -> Employee(id = "Fred", dpt = "Sales"),
    "Bert" -> Employee(id = "Bert", dpt = "Product"),
    "Cora" -> Employee(id = "Cora", dpt = "Research"),
    "Edna" -> Employee(id = "Edna", dpt = "Research"),
    "Drew" -> Employee(id = "Drew", dpt = "Research")
  ),
  tasks = HashMap(
    "program" -> Task(id = "program", hours = 3),
    "design" -> Task(id = "design", hours = 2),
    "abstract" -> Task(id = "abstract", hours = 5),
    "build" -> Task(id = "build", hours = 3),
    "call" -> Task(id = "call", hours = 1)
  ),
  knows = List(
    ("Alex", "build"),
    ("Bert", "build"),
    ("Cora", "abstract"),
    ("Cora", "build"),
    ("Cora", "design"),
    ("Drew", 

Los modelos de datos planos están en realidad muy cerca de los modelos de datos _relacionales_ habituales usados en los almacenes persistentes SQL. Este es el modelo relacional equivalente de la base de datos de la organización:

![](../../images/relational-model.png)

Según esta correspondencia:
- La clase `Organization` representa toda la _base de datos_ relacional.
- Los miembros de esta clase corresponden a las distintas _tablas_ de la base de datos, representadas como `Map`s o `Set`s simples. Tenemos cuatro tablas: la tabla de departamentos, empleados, tareas, y una tabla que almacena qué tareas pueden realizar los empleados.
- El tipo de la clave de `Map` puede entenderse como la clave primaria de la tabla relacional. El tipo del valor especifica las columnas de la tabla. Por convención, el tipo del identificador se define mediante el alias de tipo `Id` en el objeto compañero del tipo del valor. Por ejemplo, la tabla `employees` está indexada por el identificador del empleado (un valor de tipo cadena), y almacena el departamento al que pertenece el empleado.
- Si la clave primaria consiste en varias claves, como en la tabla `knows`, usamos tuplas.
- Si la tabla consiste únicamente en la clave (simple o compuesta) usamos `Set` en vez de `Map` (como también ilustra la tabla `knows`).

## Consultas básicas

Las consultas complejas habitualmente se construyen a partir de consultas básicas directamente relacionadas con la estructura del modelo de datos. En particular, se identifican a partir de la clave primaria y las relaciones de clave foránea en el modelo relacional. En la base de datos de la organización podemos identificar las siguientes consultas:

In [6]:

object BasicQueries:

    // Entidades
    
    def departments(org: Organization): List[Department] = 
        org.departments.values.toList

    def departmentIds(org: Organization): List[Department.Id] = 
        org.departments.keys.toList
    
    def getDepartment(id: Department.Id)(org: Organization): List[Department] = 
        org.departments.get(id).toList

    def employees(org: Organization): List[Employee] = 
        org.employees.values.toList

    def employeeIds(org: Organization): List[Employee.Id] = 
        org.employees.keys.toList

    def getEmployee(id: Employee.Id)(org: Organization): List[Employee] = 
        org.employees.get(id).toList

    def tasks(org: Organization): List[Task] = 
        org.tasks.values.toList

    def taskIds(org: Organization): List[Task.Id] = 
        org.tasks.keys.toList

    def getTask(id: Task.Id)(org: Organization): List[Task] = 
        org.tasks.get(id).toList
    
    // relaciones 1-N
    
    def employeeIds(dpt: Department.Id)(org: Organization): List[Employee.Id] = 
        org.employees.filter(_._2.dpt == dpt).map(_._1).toList
    
    // relaciones N-M
    
    def capabilities(emp: Employee.Id)(org: Organization): List[Task.Id] = 
        org.knows.filter(_._1 == emp).map(_._2)

    def performerIds(tsk: Task.Id)(org: Organization): List[Employee.Id] = 
        org.knows.filter(_._2 == tsk).map(_._1)

import BasicQueries._

defined object BasicQueries
import BasicQueries._


## Consultas de ejemplo

__¿Cuáles son las tareas de la organización que no puede realizar ningún empleado?__

In [ ]:
class TestImpossibleTasks(
    impossibleTasks: Organization => List[Task.Id]
) extends AnyFlatSpec with should.Matchers:
    
    "impossibleTasks" should "work" in:
        impossibleTasks(org) shouldBe 
            List("program")

Esta es una implementación imperativa convencional, usando variables mutables:

In [ ]:
import collection.mutable.ListBuffer

def impossibleTasks(org: Organization): List[Task.Id] =
    var impTasks: ListBuffer[Task.Id] = 
        ListBuffer.from(taskIds(org))
    for (entry <- org.knows)
        impTasks -= entry._2
//        impTasks = impTasks subtractOne entry._2
    impTasks.toList

In [ ]:
run(TestImpossibleTasks(impossibleTasks))

Esto funciona, pero no es el estilo _funcional_. La siguiente versión está más cerca de lo que buscamos:

In [ ]:
def impossibleTasks(org: Organization): List[Task.Id] =
    org.knows.foldLeft(ListBuffer.from(taskIds(org))):
        (impTasks, entry) => impTasks subtractOne entry._2
    .toList

o con sintaxis de pattern matching:

In [ ]:
def impossibleTasks(org: Organization): List[Task.Id] =
    org.knows.foldLeft(ListBuffer.from(taskIds(org))):
        case (impTasks, (_, task)) => impTasks subtractOne task
    .toList

In [ ]:
run(TestImpossibleTasks(impossibleTasks))

Pero podemos hacerlo todavía mejor. Adoptaremos la siguiente implementación, que usa operaciones de conjuntos de alto nivel (`diff`) y HOF (`map`):

In [ ]:
def impossibleTasks(org: Organization): List[Task.Id] =
    val possibleTasks = org.knows.map(entry => entry._2)
    taskIds(org) diff possibleTasks

In [ ]:
run(TestImpossibleTasks(impossibleTasks))

Podría decirse que esta implementación transmite la intención de la función con más claridad. Es más _declarativa_. Además, es más fiable, ya que se apoya en métodos estándar de la biblioteca de Scala (`diff` y `map`). Es cierto que la versión imperativa también es fácil de leer, pero eso es solo porque esta es una función muy simple. Más adelante veremos ejemplos más complejos donde la solución funcional brilla más.

__¿Qué tareas pueden realizar los empleados de un departamento dado?__

In [ ]:
class TestAllTasks(
    allTasks: Department.Id => Organization => List[Task.Id]
) extends AnyFlatSpec with should.Matchers:
    
    "allTasks" should "work" in:
        allTasks("Product")(org).toSet shouldBe 
            Set("build")
        allTasks("Quality")(org).toSet shouldBe 
            Set()
        allTasks("Sales")(org).toSet shouldBe 
            Set("call")
        allTasks("Research")(org).toSet shouldBe 
            Set("abstract", "build", "design", "call")

Las consultas básicas del modelo de datos nos permiten obtener todos los empleados de una organización, y las tareas que pueden realizar. Así, este es un primer paso hacia la solución:

In [ ]:
// def allTasks(dpt: Department.Id)(org: Organization): List[Task.Id]
def allTasks(dpt: Department.Id)(org: Organization): List[List[Task.Id]] = 
    employeeIds(dpt)(org) map:
        emp => capabilities(emp)(org)

Sin embargo, esta no es la signatura que necesitamos implementar, ya que estamos devolviendo un conjunto de conjuntos de tareas, no un conjunto de tareas. Para hacerlo bien también necesitamos _aplanar_ (flatten) el resultado, es decir, concatenar todos los conjuntos individuales de tareas de cada empleado. En resumen, necesitamos la HOF `flatMap`:

In [ ]:
def allTasks(dpt: Department.Id)(org: Organization): List[Task.Id] = 
    employeeIds(dpt)(org).flatMap:
        emp => capabilities(emp)(org)
    .distinct

In [ ]:
run(TestAllTasks(allTasks))

__Calcula la lista de departamentos de una organización junto con el número de tareas que pueden realizar sus empleados, ordenada por el número de tareas__

In [ ]:
class TestSortedDeps(
    sortedDeps: Organization => List[(Department.Id, Int)]
) extends AnyFlatSpec with should.Matchers:
    
    "sortedDeps" should "work" in:
        sortedDeps(org).toSet shouldBe 
            Set(("Research",4), ("Product",1), ("Sales",1), ("Quality",0))

In [ ]:
def sortedDeps(org: Organization): List[(Department.Id, Int)] = 
    departmentIds(org).map: dep => 
        (dep, allTasks(dep)(org).size)
    .sortWith: (tuple1, tuple2) => 
        tuple1._2 > tuple2._2

o con sintaxis de pattern matching:

In [ ]:
def sortedDeps(org: Organization): List[(Department.Id, Int)] = 
    departmentIds(org).map: dep => 
        (dep, allTasks(dep)(org).size)
    .sortWith: 
        case ((_, n1), (_, n2)) => n1 > n2

In [ ]:
run(TestSortedDeps(sortedDeps))

__¿Cuáles son los empleados que pueden realizar tareas de una duración dada?__

In [ ]:
class TestPersistentEmps(
    persistentEmps: Int => Organization => List[Employee.Id]
) extends AnyFlatSpec with should.Matchers:
    
    "persistentEmps" should "work" in:
        persistentEmps(3)(org).toSet shouldBe 
            Set("Cora", "Drew", "Edna")

In [ ]:
def persistentEmps(min: Int)(org: Organization): List[Employee.Id] = 
    org.knows.flatMap:
        case (emp, task) => 
            getTask(task)(org).filter: task =>
                task.hours > min
            .map: _ => 
                emp
    .distinct

Alternativamente, también podemos apostar por `filter` y `exists`:

In [ ]:
def persistentEmps(min: Int)(org: Organization): List[Employee.Id] = 
    employeeIds(org).filter: empId =>  
        capabilities(empId)(org).flatMap: taskId => 
            getTask(taskId)(org)
        .exists: task => 
            task.hours > min

In [ ]:
run(TestPersistentEmps(persistentEmps))

__¿Cuáles son los departamentos cuyos empleados, como equipo, saben realizar un conjunto de tareas dado?__

In [ ]:
class TestDptsThatKnowHowTo(
    dptsThatKnowHowTo: Set[Task.Id] => Organization => List[Department.Id]
) extends AnyFlatSpec with should.Matchers:
    
    "dptsThatKnowHowTo" should "work" in:
        dptsThatKnowHowTo(Set())(org).toSet shouldBe 
            Set("Sales", "Product", "Quality", "Research")
        dptsThatKnowHowTo(Set("call"))(org).toSet shouldBe 
            Set("Sales", "Research")
        dptsThatKnowHowTo(Set("call", "abstract"))(org).toSet shouldBe 
            Set("Research")

Podemos apoyarnos en la función anterior `allTasks`:

In [ ]:
def dptsThatKnowHowTo(tasks: Set[Task.Id])(org: Organization): List[Department.Id] = 
    departmentIds(org).filter: dpt => 
        tasks subsetOf allTasks(dpt)(org).toSet

In [ ]:
run(TestDptsThatKnowHowTo(dptsThatKnowHowTo))

__Obtén una lista de empleados ordenada por el número de tareas que pueden realizar__

In [ ]:
class TestSortedEmployees(
    sortedEmployees: Organization => List[(Employee.Id, Int)]
) extends AnyFlatSpec with should.Matchers:
    
    "sortedEmployees" should "work" in:
        sortedEmployees(org) shouldBe 
            List(
              ("Alex", 1),
              ("Fred", 1),
              ("Bert", 1),
              ("Drew", 2),
              ("Cora", 3),
              ("Edna", 3))

Podríamos intentar lo siguiente:

In [ ]:
def sortedEmployees(org: Organization): List[(Employee.Id, Int)] = 
    org.knows
        .groupBy(_._1)     // Map[Employee.Id, List[Task.Id]]
        .view.mapValues(_.size) // Map[Employee.Id, Int]
        .toList            // List[(Employee.Id, Int)]
        .sortBy(_._2) 

y esto es casi correcto: nos faltan aquellos empleados que no pueden realizar ninguna tarea. Esta es la correcta:

In [ ]:
def sortedEmployees(org: Organization): List[(Employee.Id, Int)] = 
    employeeIds(org).map: empId => 
        (empId, capabilities(empId)(org).size)
    .sortBy(_._2)

In [ ]:
run(TestSortedEmployees(sortedEmployees))

__¿Cuáles son los departamentos cuyos empleados son todos capaces de realizar una tarea dada?__

In [ ]:
class TestExpertDepsIn(
    expertDpts: Task.Id => Organization => List[Department.Id]
) extends AnyFlatSpec with should.Matchers:
    
    "expertDpts" should "work" in:
        expertDpts("abstract")(org).toSet shouldBe 
            Set("Quality", "Research")

La solución imperativa convencional es bastante compleja:

In [ ]:
def expertDepsIn(task: Task.Id)(org: Organization): List[Department.Id] =
    var out: ListBuffer[Department.Id] = ListBuffer()
    for (dep <- departmentIds(org))
        var employeesThatCantPerform: ListBuffer[Employee.Id] = 
            ListBuffer.from(employeeIds(dep)(org))
        for (emp <- employeeIds(dep)(org))
            for (someTask <- capabilities(emp)(org))
                if (someTask == task)
                    employeesThatCantPerform -= emp
        if (employeesThatCantPerform.isEmpty) 
            out += dep
    out.toList

In [ ]:
run(new TestExpertDepsIn(expertDepsIn))

Esto no solo es más complejo de entender, sino propenso a errores. Para obtener una solución más simple (y funcional) empecemos declarando en lenguaje natural llano la consulta pretendida:

In [ ]:
def expertDepsIn(tsk: Task.Id)(org: Organization): List[Department.Id] = 
    // De entre todos los departamentos de la organización, elige
    // aquellos en los que, para todos sus empleados,
    // la tarea especificada está incluida en sus capacidades
    ???

Entonces, podemos formalizar la especificación en lenguaje natural apoyándonos en HOF estándar (`filter`, `forall`) y operaciones de colecciones (`contains`):

In [ ]:
def expertDepsIn(tsk: Task.Id)(org: Organization): List[Department.Id] = 
    departmentIds(org).filter:
        dpt => employeeIds(dpt)(org).forall:
            emp => capabilities(emp)(org).contains(tsk)

In [ ]:
run(TestExpertDepsIn(expertDepsIn))